# LinguoAI – Google-Colab-Version

Diese Ausgabe startet eine passwortgeschützte Weboberfläche. Videos, Stimmreferenzen und Piper-Modelle können direkt hochgeladen oder sicher aus `MyDrive` gelesen werden. NLLB-200 und Faster-Whisper laufen lokal auf GPU oder CPU. Für die Sprachausgabe stehen Edge TTS (online), Piper (lokales Stimmenmodell) und Chatterbox (lokale Stimmreferenz) zur Wahl.

Optional: **Laufzeit → Laufzeittyp ändern → T4 GPU**. CPU funktioniert, Chatterbox ist darauf aber deutlich langsamer. Wähle das benötigte lokale TTS-Paket in Zelle 3; `none` hält den Basisstart klein und Edge TTS funktioniert weiterhin. Nutze eine Stimmreferenz nur mit ausdrücklicher Erlaubnis. Das NLLB-Standardmodell steht unter CC-BY-NC-4.0 und ist ein Forschungsmodell, keine zertifizierte Produktionsübersetzung. `/content` ist flüchtig; Ergebnisse herunterladen oder nach Drive kopieren.

In [ ]:
# @title 1. Projektquelle und Google Drive
import os

PROJECT_SOURCE = "upload_zip"  # @param ["upload_zip", "git"]
GIT_URL = ""  # @param {type:"string"}
GIT_REF = ""  # @param {type:"string"}
MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
CACHE_MODELS_IN_DRIVE = True  # @param {type:"boolean"}

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
if CACHE_MODELS_IN_DRIVE and MOUNT_GOOGLE_DRIVE:
    cache = "/content/drive/MyDrive/LinguoAI/model-cache"
    os.makedirs(cache, exist_ok=True)
    os.environ["HF_HOME"] = cache
    os.environ["LINGUOAI_MODEL_CACHE"] = cache
print("Modellcache:", os.environ.get("HF_HOME", "/content (temporär)"))

In [ ]:
# @title 2. Projekt sicher laden
import shutil
import stat
import subprocess
import zipfile
from pathlib import Path

from google.colab import files

PROJECT_AREA = Path("/content/linguoai-project")
if PROJECT_AREA.exists():
    shutil.rmtree(PROJECT_AREA)
PROJECT_AREA.mkdir(parents=True)


def safe_extract_zip(archive_path: Path, destination: Path) -> None:
    root = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        entries = archive.infolist()
        if len(entries) > 10000 or sum(item.file_size for item in entries) > 500 * 1024**2:
            raise ValueError("Projekt-ZIP ist ungewöhnlich groß.")
        for info in entries:
            member = Path(info.filename)
            target = (root / member).resolve()
            mode = (info.external_attr >> 16) & 0o170000
            if member.is_absolute() or ".." in member.parts or not target.is_relative_to(root):
                raise ValueError(f"Unsicherer ZIP-Pfad: {info.filename}")
            if mode == stat.S_IFLNK:
                raise ValueError(f"Symlink im ZIP ist nicht erlaubt: {info.filename}")
        archive.extractall(root)


if PROJECT_SOURCE == "git":
    if not GIT_URL.strip():
        raise ValueError("GIT_URL fehlt.")
    repository = PROJECT_AREA / "repo"
    subprocess.run(["git", "clone", "--filter=blob:none", GIT_URL, str(repository)], check=True)
    if GIT_REF.strip():
        subprocess.run(
            ["git", "-C", str(repository), "fetch", "--depth", "1", "origin", GIT_REF],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(repository), "checkout", "--detach", "FETCH_HEAD"],
            check=True,
        )
else:
    print("Bitte das LinguoAI-Projekt als ZIP hochladen.")
    uploaded = files.upload()
    archives = [(name, data) for name, data in uploaded.items() if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Bitte genau ein Projekt-ZIP hochladen.")
    archive_path = PROJECT_AREA / "project.zip"
    archive_path.write_bytes(archives[0][1])
    extraction = PROJECT_AREA / "uploaded"
    extraction.mkdir()
    safe_extract_zip(archive_path, extraction)

candidates = [
    p.parent for p in PROJECT_AREA.rglob("pyproject.toml") if (p.parent / "linguoai").is_dir()
]
if len(candidates) != 1:
    raise RuntimeError(f"Projektwurzel nicht eindeutig gefunden: {candidates}")
PROJECT_ROOT = candidates[0]
print("Projekt:", PROJECT_ROOT)

In [ ]:
# @title 3. Abhängigkeiten installieren und Hardware prüfen
import importlib
import shutil
import subprocess
import sys

LOCAL_TTS_EXTRA = "none"  # @param ["none", "piper", "chatterbox"]

if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
extras = ["colab"]
if LOCAL_TTS_EXTRA == "piper":
    extras.append("piper")
elif LOCAL_TTS_EXTRA == "chatterbox":
    extras.append("voice-clone")
else:
    LOCAL_TTS_EXTRA = "none"
install_target = f"{PROJECT_ROOT}[{','.join(extras)}]"
print("Installiere:", install_target)
if LOCAL_TTS_EXTRA == "chatterbox":
    print("Chatterbox ist ein großes Zusatzpaket; die Installation kann einige Minuten dauern.")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", install_target],
    check=True,
)
if LOCAL_TTS_EXTRA == "piper" and shutil.which("nvidia-smi"):
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-q", "-y", "onnxruntime", "onnxruntime-gpu"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu>=1.20,<2"],
        check=True,
    )
importlib.invalidate_caches()
torch = importlib.import_module("torch")

print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {properties.name} ({properties.total_memory / 2**30:.1f} GiB)")
else:
    print("Keine GPU erkannt - Whisper, NLLB und lokale TTS verwenden CPU.")
print("Lokales TTS-Paket:", LOCAL_TTS_EXTRA)
print(f"Freier Speicher: {shutil.disk_usage('/content').free / 2**30:.1f} GiB")

## 4. Weboberfläche starten

Die nächste Zelle zeigt Benutzername und ein zufälliges Passwort. Der erzeugte Gradio-Link ist damit geschützt. In der Oberfläche hat ein direkter Upload Vorrang vor dem jeweiligen Drive-Pfad. Drive-Pfade sind relativ zu `MyDrive`, zum Beispiel `Videos/demo.mp4` oder `LinguoAI/voices/referenz.wav`. Die Weboberfläche installiert beim Wechsel der TTS-Engine keine Pakete nach; starte Zelle 3 erneut mit dem passenden `LOCAL_TTS_EXTRA`.

In [ ]:
from linguoai.colab_ui import launch_colab

launch_colab()